# 🌍 Khởi động Dự án từ GitHub trên Google Colab

Notebook này sẽ clone mã nguồn từ GitHub thẳng vào môi trường máy ảo của Colab (`/content/`) và đẩy toàn bộ code ra đường dẫn gốc.

In [ ]:
import tqdm.notebook as tqdm
import sys
sys.modules['tqdm'] = tqdm

In [ ]:
# 1. Xóa các tệp rác mặc định của Colab (nếu có) để tạo không gian sạch
%cd /content
!rm -rf sample_data

# 2. Clone dự án từ GitHub (Thay link bằng link dự án của bạn)
!git clone https://github.com/GnurTinz/DAP_391m repo_tmp

# 3. Đẩy toàn bộ code từ thư mục clone ra đường dẫn chính (/content/)
!mv repo_tmp/* /content/
!mv repo_tmp/.* /content/ 2>/dev/null || true  # Di chuyển cả file ẩn (như .gitignore)
!rm -rf repo_tmp

print("✅ Đã tải và đẩy mã nguồn ra đường dẫn chính:")
!ls -la

In [ ]:
# 4. Cài đặt môi trường (Các thư viện theo requirements.txt)
# !pip install -r requirements.txt
!pip install tensorboard tqdm

In [ ]:
# 5. (Tùy chọn) Kết nối Drive ĐỂ LƯU LOGS / CHECKPOINT
# LƯU Ý: Code chạy trên máy ảo Colab, nhưng bạn nên đồng bộ thư mục logs/ vào Drive để không mất kết quả train.
from google.colab import drive
drive.mount('/content/drive')

# Tạo symlink (đường dẫn ảo) nối thư mục logs/ của dự án với Drive
!mkdir -p /content/drive/MyDrive/palm_logs
!rm -rf /content/logs
!ln -s /content/drive/MyDrive/palm_logs /content/logs
print("✅ Mọi kết quả huấn luyện (trong thư mục logs/) sẽ được tự động lưu về Google Drive!")

## 🚀 Tiến hành Huấn Luyện hoặc Sinh Ảnh

In [ ]:
# Huấn luyện U-Net
!python -m tools.train_generative_model --config config/mnist_unet.yaml

In [ ]:
# Huấn luyện mô hình với dataset là Tongji (chế độ mixed)
# Lưu ý truyền đúng đường dẫn thư mục dataset nếu nó khác data/Tongji
!python tools/train_lightning.py \
    dataset=tongji_mixed \
    dataset.data_dir=/content/data/Tongji \
    model=unet_mock \
    training.epochs=50 \
    training.log_interval=20 \
    loss_schedules=contrastive_first

In [ ]:
# (Ví dụ khác) Huấn luyện mô hình với dataset là IITD (chế độ chia theo tay trái/phải)
# !python tools/train_lightning.py \
#     dataset=iitd_hand \
#     dataset.data_dir=/content/data/IITD \
#     model=unet_mock \
#     logging=colab \
#     training.epochs=50

## 🚀 Tiến hành Huấn Luyện (Sử dụng hệ thống Hydra)
Hệ thống hiện tại sử dụng Hydra để cấu hình lệnh linh hoạt. Bạn có thể thay đổi dataset (`tongji_session`, `tongji_mixed`, `iitd_hand`, vv) và cấu hình ngay trên câu lệnh.

In [ ]:
# Huấn luyện mô hình với dataset là Tongji (chế độ mixed)
# Lưu ý truyền đúng đường dẫn thư mục dataset nếu nó khác data/Tongji
!python tools/train_lightning.py \
    dataset=tongji_mixed \
    dataset.data_dir=/content/data/Tongji \
    model=unet_mock \
    training.epochs=50 \
    training.log_interval=20